# ⚙️ Day 3: Writing & Validating Training Pipelines
### **Project:** SME Daily Business Assistant (`SME-Daily-Business`)
### **Assignee:** Deepana Nirmal | **Jira Task:** `KAN-21`
### **Target Models:** Qwen 2.5-7B Instruct & Llama 3 8B Instruct
### **Hardware:** Google Colab Tesla T4 GPU (15GB VRAM)

## 1. Mount Google Drive & Environment Setup

In [ ]:
import os, sys, json, torch

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/AI_SME_Project'
except Exception:
    PROJECT_ROOT = './AI_SME_Project'

print(f"Project Root : {PROJECT_ROOT}")
if torch.cuda.is_available():
    print(f"✅ GPU  : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM : {torch.cuda.get_device_properties(0).total_memory/(1024**3):.1f} GB")
    print(f"   BF16 : {'supported' if torch.cuda.is_bf16_supported() else 'NOT supported (T4) — AMP fp16 disabled, using bnb compute dtype'}")
else:
    print("⚠️ No GPU! Go to Runtime → Change runtime type → T4 GPU")

In [ ]:
!pip install -q -U bitsandbytes transformers accelerate peft trl datasets
import trl, transformers, peft
print(f"TRL {trl.__version__} | Transformers {transformers.__version__} | PEFT {peft.__version__}")
print("✅ Done. First run? → Runtime → Restart runtime → then re-run all cells.")

## 2. Hugging Face Authentication

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    print("✅ Logged in via Colab Secret HF_TOKEN!")
except Exception:
    print("Continuing with public access.")

## 3. Load Processed SME Datasets

In [ ]:
train_path = os.path.join(PROJECT_ROOT, 'data', 'processed', 'train_v1.json')
val_path   = os.path.join(PROJECT_ROOT, 'data', 'processed', 'val_v1.json')

with open(train_path, 'r', encoding='utf-8') as f: train_data = json.load(f)
with open(val_path,   'r', encoding='utf-8') as f: val_data   = json.load(f)

print(f"✅ {len(train_data):,} train | {len(val_data):,} val samples.")

## 4. Test Training Pipeline — Qwen 2.5-7B (50-sample subset, 3 epochs)

In [ ]:
import gc
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

torch.cuda.empty_cache(); gc.collect()

model_id        = "Qwen/Qwen2.5-7B-Instruct"
test_output_dir = os.path.join(PROJECT_ROOT, 'models', 'checkpoints', 'test_qwen_pipeline')
os.makedirs(test_output_dir, exist_ok=True)

# ── 1. Tokenizer ─────────────────────────────────────────────────────────────
print("1. Initializing Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ── 2. Build 50-sample subset ─────────────────────────────────────────────────
def format_chatml(ex):
    sys_msg = "You are an expert SME daily business assistant."
    user_q  = ex['instruction']
    if ex.get('context'):
        user_q = f"Context:\n{ex['context']}\n\nQuestion:\n{ex['instruction']}"
    return tokenizer.apply_chat_template(
        [{"role":"system","content":sys_msg},
         {"role":"user","content":user_q},
         {"role":"assistant","content":ex['response']}],
        tokenize=False
    )

train_ds = Dataset.from_dict({"text": [format_chatml(x) for x in train_data[:50]]})
val_ds   = Dataset.from_dict({"text": [format_chatml(x) for x in val_data[:10]]})

# ── 3. Load model 4-bit NF4 ──────────────────────────────────────────────────
# NOTE: bnb_4bit_compute_dtype=float16 means bitsandbytes itself computes in
# float16 internally. We do NOT need PyTorch AMP (fp16=True) on top of this.
# Enabling AMP on T4 causes the BFloat16 grad-scaler crash → we disable it.
print("2. Loading Base Model in 4-bit NF4...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16   # bnb handles float16; no AMP needed
)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)
model.config.use_cache = False

# ── 4. PEFT / LoRA ───────────────────────────────────────────────────────────
print("3. Applying LoRA Adapters...")
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                    "gate_proj","up_proj","down_proj"]
))
model.print_trainable_parameters()

# ── 5. SFTConfig — AMP disabled to avoid T4 BFloat16 grad-scaler crash ───────
print("4. Setting Training Arguments...")
sft_config = SFTConfig(
    output_dir=test_output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,       # reduced to fit T4 VRAM without AMP
    gradient_accumulation_steps=8,       # effective batch = 16 (same as before)
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=2,
    # ↓↓↓ KEY FIX: Both AMP modes OFF ↓↓↓
    # T4 has no BFloat16 support → fp16 AMP scaler crashes on BF16 bnb tensors.
    # bitsandbytes already handles float16 computation internally.
    fp16=False,
    bf16=False,
    logging_steps=2,
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    max_length=512,
    optim="paged_adamw_32bit",
    seed=42
)

# ── 6. SFTTrainer & Train ────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    args=sft_config
)

print("\n5. Running Test Training (3 Epochs)...\n")
trainer.train()
print("\n✅ Qwen 2.5-7B Training Pipeline Verified Successfully!")

del model, tokenizer, trainer
torch.cuda.empty_cache(); gc.collect()

## 5. Day 3 Verification & Metadata Save

In [ ]:
metadata = {
    "Day": "Day 3 - Writing Training Pipelines",
    "Jira_Task": "KAN-21",
    "Engineer": "Deepana Nirmal",
    "Domain": "SME Daily Business",
    "Pipelines_Built": ["src/training/train_qwen.py", "src/training/train_llama.py"],
    "Pipeline_Specs": {
        "Trainer": "SFTTrainer (TRL 1.x)",
        "Quantization": "4-bit NF4 + Double Quant (bnb float16 compute)",
        "AMP": "Disabled (fp16=False, bf16=False) — T4 has no BFloat16 support",
        "LoRA_Rank": 16, "LoRA_Alpha": 32, "LoRA_Dropout": 0.05,
        "Optimizer": "paged_adamw_32bit",
        "Effective_Batch_Size": 16, "Learning_Rate": 2e-4, "Scheduler": "Cosine"
    },
    "Status": "TEST_PIPELINE_VERIFIED_CONVERGING"
}

meta_path = os.path.join(PROJECT_ROOT, 'day3_metadata.json')
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)

print("✅ Day 3 (KAN-21) Completed and Verified!")
print(json.dumps(metadata, indent=2))
print("\n🎉 Ready for Day 4: Running v1 Training (KAN-26)!")